# 03 · Vector search: brute force, FAISS, IVF, HNSW and PQ

After this notebook you can do exact cosine search in NumPy, reproduce it with FAISS, build IVF, HNSW and product-quantised indexes, **measure** the recall/latency trade-off of each dial (`nprobe`, `efSearch`), size an index from memory arithmetic, and explain why post-filtering silently loses results.

**Time:** ~30 min · **Runs:** offline on CPU in ~1 min · **Needs:** [embeddings and similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb), [BM25 from scratch](02_bm25_from_scratch.ipynb)

## Why this matters in interviews

- "You have 10 million vectors - how do you find the nearest without comparing all of them?" is a standard question. The answer is ANN (HNSW or IVF), and the follow-up is **"what do you give up?"** - recall, which you will measure against exact search.
- Interviewers want the dials by name: `M` / `efSearch` for HNSW, `nlist` / `nprobe` for IVF, `m` sub-vectors for PQ - and the **memory arithmetic** that decides the index at scale.
- Filtered search is where production systems fail silently: post-filtering returns fewer than k results and nothing errors. You will reproduce it.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ve01` | What is cosine similarity, and what are the alternatives? |
| `ve23` | What is normalisation and when must you do it? |
| `ve03` | Your vector database has 10 million documents. How does it find the nearest without comparing all 10 million? |
| `ve04` | How does HNSW work? |
| `ve05` | How does IVF work, and how does it compare to HNSW? |
| `ve07` | What is the recall/latency trade-off in ANN, and how do you tune it? |
| `ve06` | What is product quantisation? |
| `ve30` | You have 1.2 billion vectors and 64 GB of RAM per node. What decides your index? |
| `ve16` | How would you handle filtered vector search efficiently? |
| `rg33` | What is a vector database and do you always need one? |

## Setup

Two datasets: the **real** Acme knowledge base (42 section chunks, MiniLM embeddings) for semantics, and **synthetic** vectors (20,000 × 64) for the speed/recall curves - big enough for ANN to matter, small enough to run in seconds.

In [ ]:
import os
import tempfile
import time

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

faiss.omp_set_num_threads(1)          # single thread: deterministic index builds and cleaner per-query timings
rng = np.random.default_rng(0)
pd.set_option("display.width", 150)
print("faiss", faiss.__version__)

In [ ]:
# The knowledge base: 15 short markdown docs for a fictional company, Acme Cloud (a SaaS analytics product
# that also sells an 'Edge Gateway' device). This cell is data - it is identical in every 09_rag notebook.
KB = {
"refund_policy.md": """# Refund Policy

## Eligibility
Annual plans can be refunded in full within 30 days of purchase. Monthly plans can be cancelled at any time, but the current month is not refunded. Add-on credits and professional-services hours are non-refundable.

## How to request a refund
Open the billing portal, choose the invoice and click "Request refund". Only workspace owners can request refunds.

## Processing time
Approved refunds go back to the original payment method within 5-10 business days. Bank transfers can take up to 15 business days.
""",
"shipping.md": """# Hardware Shipping

## Delivery times
Edge Gateway devices ship from our warehouses in Rotterdam and Dallas. Standard delivery takes 3-5 business days. Express delivery takes 1-2 business days and costs $25.

## Tracking
A tracking link is emailed when the order leaves the warehouse. The first carrier scan can take 24 hours to appear.

## Lost or damaged shipments
If a shipment has not arrived 10 business days after dispatch, report it within 14 days so we can open a carrier investigation. We send a free replacement when the carrier confirms the loss.
""",
"warranty.md": """# Edge Gateway Warranty

## Coverage
Every Edge Gateway has a 2-year limited hardware warranty from the delivery date. It covers manufacturing defects in the device, the power supply and the antennas.

## Not covered
Water damage, drops, unauthorised repairs and cosmetic scratches are not covered by the warranty.

## RMA process
Request an RMA (return merchandise authorisation) in the support portal with the device serial number. Once the RMA is approved we ship a replacement within 7 days, and you send the faulty unit back with the prepaid label.
""",
"api_rate_limits.md": """# API Rate Limits

## Limits per plan
Free: 60 requests per minute. Pro: 600 requests per minute. Business: 1,200 requests per minute. Enterprise: 6,000 requests per minute. Limits apply per API key.

## Best practices
Retry with exponential backoff and jitter, cache responses you read often, and send writes through the bulk endpoint.
""",
"api_errors.md": """# API Error Reference

## ERR-429-A
The per-minute rate limit was exceeded. Wait for the number of seconds in the Retry-After header, then retry.

## ERR-429-B
The daily request quota is exhausted. The quota resets at 00:00 UTC; upgrade the plan for a higher quota.

## ERR-503-C
The API is in scheduled maintenance. Retry after the maintenance window shown on the status page.

## ERR-401-D
The API key was revoked or has expired. Create a new key in Settings > API keys.
""",
"gateway_models.md": """# Edge Gateway Models

## EG-100
The EG-100 is the entry model: 2 Ethernet ports, Wi-Fi 5 and an 8 W maximum power draw. It supports up to 50 sensors.

## EG-200
The EG-200 adds LTE backup: 4 Ethernet ports, Wi-Fi 6 and a 12 W maximum power draw. It supports up to 200 sensors.

## EG-300
The EG-300 is the rugged outdoor model: an IP67 enclosure, 4 Ethernet ports, Wi-Fi 6 and an 18 W maximum power draw. It supports up to 500 sensors.
""",
"oncall_policy.md": """# On-call Policy

## Rotation
Engineers join the on-call rotation after their third month. Each shift lasts one week and hands over on Monday at 09:00 local time. Every shift has a primary and a secondary.

## Response targets
A SEV1 page must be acknowledged within 5 minutes and a SEV2 page within 30 minutes. If a SEV1 page is not acknowledged within 15 minutes, it escalates to the secondary and then to the engineering manager.

## Compensation
On-call engineers receive a stipend of $300 per week, and a day off after any night with more than two pages.
""",
"pricing.md": """# Pricing Plans

## Plans
Free costs $0 and includes 3 users and 1 GB of storage. Pro costs $49 per user per month. Business costs $99 per user per month. Enterprise pricing is custom and starts at 50 users.

## What each plan includes
Single sign-on (SAML and OIDC) is included from the Business plan upward. Audit logs and the 99.95% uptime SLA are included only in Enterprise. All paid plans include email support.

## Billing
Paying annually saves 20% compared with monthly billing. Prices exclude VAT.
""",
"sso_setup.md": """# Single Sign-On Setup

## Supported identity providers
Acme Cloud supports Okta, Microsoft Entra ID (formerly Azure AD) and Google Workspace through SAML 2.0 or OIDC.

## Configuration steps
Go to Settings > Security > Single sign-on, upload the identity provider's metadata XML, and map the email and groups attributes. Test with one account before you enforce SSO for the whole workspace.

## Troubleshooting
SSO-ERR-12 means the SAML assertion was rejected because the clocks differ by more than 5 minutes; sync the identity provider's clock with NTP. SSO-ERR-31 means the email attribute is missing from the assertion.
""",
"data_retention.md": """# Data Retention

## Event data
Raw event data is kept for 30 days on Free, 13 months on Pro and Business, and 25 months on Enterprise. Aggregated reports are kept for as long as the workspace exists.

## Deleted workspaces
When a workspace is deleted, its data is purged from primary storage after 30 days and from backups after a further 35 days.

## Erasure requests
GDPR erasure requests for one person's data are completed within 30 days of identity verification.
""",
"pto_policy.md": """# Paid Time Off

## Allowance
Full-time employees receive 25 days of paid time off per calendar year, plus public holidays. Part-time employees receive a pro-rata allowance.

## Booking
Book absences longer than three days in Workday at least two weeks ahead. Your manager approves the request.

## Carry-over
Up to 5 unused days can be carried into the next year and must be used by 31 March.
""",
"webhooks.md": """# Webhooks

## Delivery and retries
Webhook events are sent as HTTPS POST requests. If your endpoint does not return a 2xx status within 10 seconds, we retry up to 8 times with exponential backoff over 24 hours.

## Verifying signatures
Every request carries an X-Acme-Signature header: an HMAC-SHA256 of the raw body computed with your webhook secret. Compare it in constant time and reject requests older than 5 minutes.
""",
"sla.md": """# Service Level Agreement

## Uptime commitment
Pro and Business are backed by a 99.9% monthly uptime commitment. Enterprise is backed by 99.95%.

## Service credits
If monthly uptime falls below the commitment, you receive a service credit of 10% of the monthly fee; below 99.0% the credit is 25%. Credits are applied to the next invoice.

## How to claim
Open a support ticket within 30 days of the end of the affected month and include the incident dates.
""",
"account_security.md": """# Account Security

## Passwords
Password reset links expire after 60 minutes. After 5 failed sign-in attempts the account is locked for 15 minutes.

## Two-factor authentication
Workspace owners can require two-factor authentication (2FA) for every member. Authenticator apps and hardware security keys are supported; SMS codes are not.
""",
"expenses.md": """# Travel and Expenses

## Meals and hotels
Meals are reimbursed up to $75 per day while travelling. Book hotels through the travel portal; the nightly cap is $200 in most cities.

## Submitting claims
Submit receipts in Workday within 30 days of the expense. Claims over $25 without a receipt are rejected.
""",
}

In [ ]:
# The eval set: each question lists its EVIDENCE - (doc, phrase) pairs that a retrieved chunk must contain.
# Phrases are unique in the KB, so "is the evidence in the retrieved text?" works for any chunking scheme.
QUESTIONS = [
    {"q": "What does ERR-429-A mean?", "kind": "exact id", "evidence": [("api_errors.md", "per-minute rate limit")]},
    {"q": "Our integration keeps failing with ERR-401-D", "kind": "exact id", "evidence": [("api_errors.md", "revoked")]},
    {"q": "How do I fix SSO-ERR-12?", "kind": "exact id", "evidence": [("sso_setup.md", "clocks differ")]},
    {"q": "How many sensors does the EG-300 support?", "kind": "exact id", "evidence": [("gateway_models.md", "500 sensors")]},
    {"q": "How do I get my money back after buying an annual plan by mistake?", "kind": "paraphrase",
     "evidence": [("refund_policy.md", "refunded in full within 30 days")]},
    {"q": "My parcel never showed up. What should I do?", "kind": "paraphrase", "evidence": [("shipping.md", "report it within 14 days")]},
    {"q": "How many vacation days do I get a year?", "kind": "paraphrase", "evidence": [("pto_policy.md", "25 days of paid time off")]},
    {"q": "Do we get compensated for outages?", "kind": "paraphrase", "evidence": [("sla.md", "service credit")]},
    {"q": "Which gateway can be installed outdoors?", "kind": "paraphrase", "evidence": [("gateway_models.md", "rugged outdoor model")]},
    {"q": "Who gets paged when nobody acknowledges a SEV1 alert?", "kind": "keyword", "evidence": [("oncall_policy.md", "engineering manager")]},
    {"q": "How long is event data kept on the Pro plan?", "kind": "keyword", "evidence": [("data_retention.md", "13 months on Pro")]},
    {"q": "What do we get back if the platform is down more than promised?", "kind": "vague", "evidence": [("sla.md", "service credit")]},
    {"q": "What is the API rate limit on the cheapest plan that includes SSO?", "kind": "multi-hop",
     "evidence": [("pricing.md", "from the Business plan upward"), ("api_rate_limits.md", "Business: 1,200 requests")]},
    {"q": "How long do approved refunds take, and how long is the gateway warranty?", "kind": "multi-part",
     "evidence": [("refund_policy.md", "5-10 business days"), ("warranty.md", "2-year")]},
    {"q": "What is the capital of Australia?", "kind": "out-of-scope", "evidence": []},
]
IN_SCOPE = [q for q in QUESTIONS if q["evidence"]]


def has_evidence(text, doc, q):
    """True if this text (from `doc`) contains any of the question's evidence phrases."""
    return any(doc == d and p.lower() in text.lower() for d, p in q["evidence"])


print(f"{len(KB)} docs, {sum(len(t) for t in KB.values()):,} characters; "
      f"{len(QUESTIONS)} questions ({len(IN_SCOPE)} answerable, {len(QUESTIONS) - len(IN_SCOPE)} out of scope)")

In [ ]:
import re


def split_sections(kb):
    """One chunk per '## ' section. The heading path is prepended so every chunk says where it came from."""
    chunks = []
    for doc, text in kb.items():
        title = text.splitlines()[0].lstrip("# ").strip()
        for i, part in enumerate(re.split(r"^## ", text, flags=re.M)[1:]):
            heading, _, body = part.partition("\n")
            path = f"{title} > {heading.strip()}"
            chunks.append({"id": f"{doc}#{i}", "doc": doc, "section": path, "text": f"{path}\n{body.strip()}"})
    return chunks


def is_relevant(chunk, q):
    return has_evidence(chunk["text"], chunk["doc"], q)


CHUNKS = split_sections(KB)
print(f"{len(CHUNKS)} section chunks, avg {sum(len(c['text']) for c in CHUNKS) / len(CHUNKS):.0f} characters")
print("example:", CHUNKS[1]["id"], "->", CHUNKS[1]["text"][:90].replace("\n", " | "), "...")

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import transformers
transformers.utils.logging.disable_progress_bar()          # keep the cell output clean
transformers.logging.set_verbosity_error()
from sentence_transformers import SentenceTransformer


def load_st(name, cls=SentenceTransformer):
    """Load a small Hugging Face model: cached copy first (no network), download only if missing."""
    try:
        return cls(name, local_files_only=True)
    except Exception:
        try:
            return cls(name)                                 # first run only: ~90 MB download
        except Exception as e:
            raise RuntimeError(f"Could not load {name}: connect to the internet once so it can download.") from e


embedder = load_st("sentence-transformers/all-MiniLM-L6-v2")


def encode(texts):
    """Unit-length MiniLM embeddings (384-d), so a dot product IS the cosine similarity."""
    return embedder.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True)


print("embedder: all-MiniLM-L6-v2 |", embedder.get_embedding_dimension(), "dims | max",
      embedder.max_seq_length, "tokens per text (the rest is silently cut off)")

## 1. Exact search in NumPy - the baseline and the ground truth

**In plain English:** compare the question's vector with every chunk vector and keep the k most similar. With unit-length vectors, cosine similarity is just a dot product, so "search" is one matrix-vector multiply plus a partial sort.

In [ ]:
chunk_vecs = encode([c["text"] for c in CHUNKS]).astype("float32")
print("chunk matrix:", chunk_vecs.shape, "| norms ~1:", np.allclose(np.linalg.norm(chunk_vecs, axis=1), 1, atol=1e-5))


def numpy_topk(query_vecs, matrix, k):
    scores = query_vecs @ matrix.T                               # (n_queries, n_chunks) cosine similarities
    idx = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]     # top-k in O(n), unordered
    order = np.take_along_axis(scores, idx, axis=1).argsort(axis=1)[:, ::-1]
    return np.take_along_axis(idx, order, axis=1)


q_vecs = encode([q["q"] for q in QUESTIONS]).astype("float32")
top3 = numpy_topk(q_vecs, chunk_vecs, 3)
for q, row in list(zip(QUESTIONS, top3))[:3]:
    print(f"\n{q['q']}")
    for i in row:
        print(f"   {float(q_vecs[QUESTIONS.index(q)] @ chunk_vecs[i]):.3f}  {CHUNKS[i]['id']}")

Cosine, dot product and Euclidean distance give **the same ranking on unit vectors**, because $\|a - b\|^2 = 2 - 2\cos(a, b)$. Check it on every question:

In [ ]:
for qv in q_vecs:
    by_cos = np.argsort(-(chunk_vecs @ qv) / (np.linalg.norm(chunk_vecs, axis=1) * np.linalg.norm(qv)))
    by_dot = np.argsort(-(chunk_vecs @ qv))
    by_l2 = np.argsort(np.linalg.norm(chunk_vecs - qv, axis=1))
    assert (by_cos[:10] == by_dot[:10]).all() and (by_dot[:10] == by_l2[:10]).all()
a, b = chunk_vecs[0], chunk_vecs[1]
print(f"|a-b|^2 = {np.sum((a - b) ** 2):.5f}   2 - 2cos = {2 - 2 * float(a @ b):.5f}")
print("cosine, dot and L2 rankings are identical on all", len(q_vecs), "questions")

## 2. FAISS `IndexFlatIP`: the same exact search, as a library

`IndexFlatIP` = flat (no compression, no partitioning) + inner product. Inner product equals cosine **only if you normalise** - FAISS will not do it for you.

In [ ]:
flat = faiss.IndexFlatIP(chunk_vecs.shape[1])
flat.add(chunk_vecs)
D, I = flat.search(q_vecs, 3)
assert (I == top3).all(), "FAISS flat search must equal NumPy brute force"
assert np.allclose(D[:, 0], np.sum(q_vecs * chunk_vecs[I[:, 0]], axis=1), atol=1e-5)
print("IndexFlatIP top-3 ids == NumPy top-3 ids for all", len(QUESTIONS), "questions")

# The normalisation trap: vectors whose length varies (some models, or averaged vectors) break inner-product search.
raw = rng.normal(size=(1000, 32)).astype("float32") * rng.uniform(0.2, 3.0, size=(1000, 1)).astype("float32")
query = rng.normal(size=(1, 32)).astype("float32")
exact_cos = np.argsort(-(raw @ query[0]) / np.linalg.norm(raw, axis=1))[:10]
ip_raw = faiss.IndexFlatIP(32)
ip_raw.add(raw)
_, I_raw = ip_raw.search(query, 10)
normed = raw.copy()
faiss.normalize_L2(normed)                    # in place
ip_norm = faiss.IndexFlatIP(32)
ip_norm.add(normed)
qn = query.copy()
faiss.normalize_L2(qn)
_, I_norm = ip_norm.search(qn, 10)
print(f"un-normalised IndexFlatIP agrees with cosine on {len(set(I_raw[0]) & set(exact_cos))}/10 results; "
      f"normalised: {len(set(I_norm[0]) & set(exact_cos))}/10")
assert set(I_norm[0]) == set(exact_cos) and set(I_raw[0]) != set(exact_cos)

Without normalisation, long vectors win regardless of direction. **Rule:** normalise documents at index time and queries at query time, with the same code path (MiniLM already outputs unit vectors; many models do not).

Exact search is perfect but costs O(N · d) per query. For 42 chunks that is nothing; for 10M × 768 it is ~7.7 billion multiply-adds per query. ANN indexes look at less.

## 3. A synthetic corpus for speed and recall curves

20,000 vectors in 64 dimensions drawn around 100 overlapping cluster centres (real embeddings are clumpy too), 200 queries. Ground truth = exact top-10 from brute force; **recall@10** = fraction of the true top-10 an index returns.

In [ ]:
N, DIM, NQ, K = 20_000, 64, 200, 10
centres = rng.normal(size=(100, DIM)).astype("float32")
X = (centres[rng.integers(0, 100, N)] + 1.6 * rng.normal(size=(N, DIM))).astype("float32")
Q = (centres[rng.integers(0, 100, NQ)] + 1.6 * rng.normal(size=(NQ, DIM))).astype("float32")
faiss.normalize_L2(X)
faiss.normalize_L2(Q)


def timed_search(index, queries, k, repeats=3, **params):
    """Best-of-3 wall time per query in ms, plus the ids."""
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        _, ids = index.search(queries, k, **params)
        best = min(best, time.perf_counter() - t0)
    return 1000 * best / len(queries), ids


def recall_at_k(ids, truth):
    return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(ids, truth)]))


flat_syn = faiss.IndexFlatIP(DIM)
flat_syn.add(X)
flat_ms, truth = timed_search(flat_syn, Q, K)
assert (truth == numpy_topk(Q, X, K)).all()
print(f"exact search: {flat_ms:.3f} ms/query over {N:,} vectors (this is the ground truth)")

## 4. IVF: cluster first, then search only the nearest cells

**In plain English:** run k-means to split the space into `nlist` cells; each vector lives in its nearest cell. At query time, find the `nprobe` cells whose centroids are nearest to the query and scan only those. Misses happen when a true neighbour sits just across a cell boundary in a cell you did not probe.

In [ ]:
NLIST = 128
ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(DIM), DIM, NLIST, faiss.METRIC_INNER_PRODUCT)
t0 = time.perf_counter()
ivf.train(X)                          # k-means on the vectors -> 128 centroids
ivf.add(X)
sizes = np.array([ivf.invlists.list_size(i) for i in range(NLIST)])
print(f"IVF build: {time.perf_counter() - t0:.2f}s; {NLIST} cells, sizes min/median/max = "
      f"{sizes.min()}/{int(np.median(sizes))}/{sizes.max()} vectors")

ivf_rows = []
for nprobe in [1, 2, 4, 8, 16, 32, 64, 128]:
    ivf.nprobe = nprobe
    ms, ids = timed_search(ivf, Q, K)
    ivf_rows.append({"nprobe": nprobe, "recall@10": recall_at_k(ids, truth), "ms/query": ms,
                     "fraction scanned": nprobe / NLIST})
ivf_df = pd.DataFrame(ivf_rows)
print(ivf_df.round(3).to_string(index=False))
assert (np.diff(ivf_df["recall@10"]) >= -1e-9).all(), "more probes never lose recall"
assert ivf_df["recall@10"].iloc[-1] == 1.0, "probing every cell is exhaustive search"

`nprobe` is the dial: each doubling scans twice as many vectors. Probing all 128 cells scans everything and is exact again (recall 1.0) - IVF is only fast *because* it looks at less, and only wrong for the same reason.

## 5. HNSW: a navigable graph

**In plain English:** a road network. Every vector is a node linked to about `M` near neighbours; a few nodes also appear on sparser upper "motorway" layers with long links. Search starts at the top, greedily hops toward the query, drops a layer, repeats, and on the bottom layer keeps a candidate list of size `efSearch`. Bigger `efSearch` = more nodes visited = higher recall, slower.

In [ ]:
hnsw = faiss.IndexHNSWFlat(DIM, 16, faiss.METRIC_INNER_PRODUCT)     # M = 16 links per node
hnsw.hnsw.efConstruction = 40
t0 = time.perf_counter()
hnsw.add(X)                                                         # no training: the graph is built by insertion
print(f"HNSW build: {time.perf_counter() - t0:.2f}s, top layer = {hnsw.hnsw.max_level}")

hnsw_rows = []
for ef in [8, 16, 32, 64, 128, 256]:
    hnsw.hnsw.efSearch = ef
    ms, ids = timed_search(hnsw, Q, K)
    hnsw_rows.append({"efSearch": ef, "recall@10": recall_at_k(ids, truth), "ms/query": ms})
hnsw_df = pd.DataFrame(hnsw_rows)
print(hnsw_df.round(3).to_string(index=False))
assert hnsw_df["recall@10"].is_monotonic_increasing and hnsw_df["recall@10"].iloc[-1] > 0.95

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.3))
ax.plot(ivf_df["ms/query"], ivf_df["recall@10"], "o-", color="#3b5bdb", label="IVF (label = nprobe)")
for _, r in ivf_df.iterrows():
    ax.annotate(int(r["nprobe"]), (r["ms/query"], r["recall@10"]), textcoords="offset points", xytext=(4, -10), fontsize=8, color="#3b5bdb")
ax.plot(hnsw_df["ms/query"], hnsw_df["recall@10"], "s-", color="#e8590c", label="HNSW M=16 (label = efSearch)")
for _, r in hnsw_df.iterrows():
    ax.annotate(int(r["efSearch"]), (r["ms/query"], r["recall@10"]), textcoords="offset points", xytext=(4, 4), fontsize=8, color="#e8590c")
ax.axvline(flat_ms, color="#868e96", ls="--", label="exact search (recall 1.0)")
ax.set_xscale("log")
ax.set_xlabel("ms per query (log scale, single thread)")
ax.set_ylabel("recall@10 vs exact")
ax.set_title("The ANN trade-off: every dial buys recall with latency")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

How to read it: pick the **knee** - the cheapest setting that reaches your recall target (production usually aims for 0.95-0.99 recall@10 against exact). Timings on a 20k toy set are microseconds and noisy; the *shape* is what carries over. At 20k vectors exact search is still fast (the highest `efSearch` settings here are no faster than brute force) - ANN earns its keep in the millions, where the flat line moves far to the right while the ANN curves barely move (their cost grows roughly with log N for HNSW and with N/nlist · nprobe for IVF).

| | HNSW | IVF(-PQ) |
|---|---|---|
| Structure | multi-layer proximity graph | k-means cells (inverted lists) |
| Needs training | no | yes (k-means on a sample) |
| Recall dial | `efSearch` | `nprobe` |
| Memory | vectors + ~`2M` links per node | vectors (or PQ codes) + ids |
| Updates | incremental inserts; deletes are tombstones | easy inserts; centroids drift, retrain periodically |
| Best for | ≤ tens of millions, RAM available, lowest latency | hundreds of millions+, memory-bound (with PQ) |

## 6. Product quantisation: compress the vectors

**In plain English:** chop each 64-number vector into `m = 8` pieces of 8 numbers; for each piece position learn a codebook of 256 centroids; store each piece as the 1-byte id of its nearest centroid. 256 bytes of float32 become **8 bytes**. Distances are computed from small lookup tables, and are approximate.

In [ ]:
t0 = time.perf_counter()
ivfpq = faiss.IndexIVFPQ(faiss.IndexFlatIP(DIM), DIM, 64, 8, 8, faiss.METRIC_INNER_PRODUCT)   # nlist=64, m=8, 8 bits
ivfpq.cp.niter = 10
ivfpq.pq.cp.niter = 10
ivfpq.train(X[:10_000])
ivfpq.add(X)
ivfpq.nprobe = 16
print(f"IVF-PQ build: {time.perf_counter() - t0:.2f}s, code size = {ivfpq.code_size} bytes/vector")

pq_ms, pq_ids = timed_search(ivfpq, Q, K)
pq_recall = recall_at_k(pq_ids, truth)

# The standard fix: PQ for a wide shortlist, then re-score the shortlist with the exact vectors.
_, shortlist = ivfpq.search(Q, 100)
rescored = np.array([cand[np.argsort(-(X[cand] @ q))[:K]] for cand, q in zip(shortlist, Q)])
rescored_recall = recall_at_k(rescored, truth)
print(f"IVF-PQ recall@10 alone: {pq_recall:.3f} | shortlist 100 + exact re-score: {rescored_recall:.3f}")
assert rescored_recall > pq_recall + 0.3

In [ ]:
def index_bytes(index):
    path = os.path.join(tempfile.mkdtemp(), "index.faiss")
    faiss.write_index(index, path)
    return os.path.getsize(path)


ivf.nprobe, hnsw.hnsw.efSearch = 16, 64
mem = pd.DataFrame([
    {"index": "Flat (exact)", "bytes/vector": index_bytes(flat_syn) / N, "recall@10": 1.0},
    {"index": "IVF-Flat nprobe=16", "bytes/vector": index_bytes(ivf) / N, "recall@10": recall_at_k(timed_search(ivf, Q, K)[1], truth)},
    {"index": "HNSW M=16 ef=64", "bytes/vector": index_bytes(hnsw) / N, "recall@10": recall_at_k(timed_search(hnsw, Q, K)[1], truth)},
    {"index": "IVF-PQ m=8", "bytes/vector": index_bytes(ivfpq) / N, "recall@10": pq_recall},
    {"index": "IVF-PQ m=8 + exact re-score", "bytes/vector": index_bytes(ivfpq) / N, "recall@10": rescored_recall},
])
mem["vs float32"] = (DIM * 4 / mem["bytes/vector"]).map(lambda x: f"{x:.1f}x smaller" if x > 1.05 else "1x")
print(mem.round(3).to_string(index=False))
print("(the re-score row also needs the exact vectors somewhere - usually on disk or in a separate store)")

At this toy size the PQ index is dominated by 8-byte ids and codebooks; the ratio gets closer to the ideal 32x as N grows. The arithmetic that decides real indexes (say it out loud in the interview before naming an algorithm):

In [ ]:
def gb(n_bytes):
    return n_bytes / 1024**3


for n, d in [(10_000_000, 768), (1_200_000_000, 768)]:
    print(f"{n:>13,} x {d}-d:  float32 {gb(n * d * 4):8,.0f} GB | int8 {gb(n * d):7,.0f} GB | "
          f"PQ 96 bytes {gb(n * 96):6,.0f} GB | HNSW links (M=16) +{gb(n * 2 * 16 * 4):5,.0f} GB")

10M × 768 float32 is ~29 GB - one big machine with HNSW. 1.2B vectors is ~3.4 TB raw: nothing but compression (PQ at ~96 bytes → ~107 GB) plus sharding fits a 64 GB-per-node budget, with a re-scoring pass over exact vectors on disk to win back recall.

## 7. Filtered search: pre-filter, post-filter, filtered ANN

Real queries come with filters: *this tenant*, *this product*, *updated this year*. Give every synthetic vector a `tenant` attribute and ask for the top 10 **within one tenant**.

- **Post-filter:** ANN search for the top k, then drop the ones that fail the filter. Fast - and when the filter is selective, you get back fewer than k results, often zero. **No error is raised.**
- **Post-filter with oversampling:** ask ANN for 10× k, then filter. Helps, until the filter is more selective than the oversampling factor.
- **Pre-filter:** restrict to matching vectors, then search them (exact search over the subset here). Always correct; cheap when the subset is small.
- **Filtered ANN:** push the predicate into the index traversal (FAISS `IDSelector`; Qdrant, Weaviate, Milvus, pgvector do this natively).

In [ ]:
hnsw.hnsw.efSearch = 64
rows = []
for selectivity in [0.5, 0.2, 0.05, 0.01, 0.002]:
    allowed = np.flatnonzero(rng.random(N) < selectivity)            # ids of this tenant's vectors
    allowed_set = set(allowed.tolist())
    sub_truth = allowed[numpy_topk(Q, X[allowed], K)]                # exact top-10 inside the tenant
    _, post = hnsw.search(Q, K)
    _, post10x = hnsw.search(Q, 10 * K)
    params = faiss.SearchParametersHNSW(sel=faiss.IDSelectorBatch(allowed), efSearch=64)
    _, filt = hnsw.search(Q, K, params=params)
    for name, res in [("post-filter", post), ("post-filter, 10x oversample", post10x),
                      ("filtered ANN (IDSelector)", filt), ("pre-filter + exact", sub_truth)]:
        kept = [[i for i in row if i in allowed_set][:K] for row in res]
        rows.append({"selectivity": selectivity, "method": name, "results returned (of 10)": np.mean([len(r) for r in kept]),
                     "recall@10": np.mean([len(set(r) & set(t)) / K for r, t in zip(kept, sub_truth)])})
filt_df = pd.DataFrame(rows)
print(filt_df.pivot(index="selectivity", columns="method", values="results returned (of 10)").round(2).to_string())
low = filt_df[filt_df["selectivity"] == 0.01].set_index("method")
assert low.loc["post-filter", "results returned (of 10)"] < 1.0
assert low.loc["pre-filter + exact", "results returned (of 10)"] == 10 and low.loc["pre-filter + exact", "recall@10"] == 1.0

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for (name, grp), style in zip(filt_df.groupby("method", sort=False), ["o-", "s-", "^-", "d--"]):
    ax.plot(grp["selectivity"] * 100, grp["results returned (of 10)"], style, label=name)
ax.set_xscale("log")
ax.invert_xaxis()
ax.set_xlabel("% of the corpus that passes the filter (log scale; more selective to the right)")
ax.set_ylabel("results returned (asked for 10)")
ax.set_title("The post-filter trap: selective filters silently return fewer than k")
ax.legend()
plt.tight_layout()
plt.show()

With a filter that matches 1% of the corpus, plain post-filtering returns less than one result on average - and the caller asked for ten. This is how "the answers got worse for small customers" happens in production. Oversampling only moves the cliff. Filtered ANN keeps k results for moderate filters, but look at its left end: at 0.2% the graph walk runs out of allowed neighbours to hop through and returns only a couple of results - a selective predicate breaks the graph's connectivity. That is why good stores switch to **pre-filter + brute force** when the matching subset is small: over a few thousand vectors it is exact and genuinely the fastest option. Also note: **tenant filters are a security control** - they must be enforced server-side, never as an optional ranking hint.

## 8. The vector-database landscape

You rarely call FAISS directly in production; you run a store that wraps an index with persistence, filters, updates and replication.

| Option | What it is | Choose it when | Watch out for |
|---|---|---|---|
| **FAISS** | C++/Python library (Meta): Flat, IVF, HNSW, PQ, GPU | you want raw speed, research, or to embed an index in your own service | no persistence server, no metadata filters beyond ids, you build the rest |
| **Chroma** | embedded/open-source store, Python-first | prototypes, notebooks, single-node apps | not built for large multi-tenant production |
| **pgvector** | Postgres extension (HNSW / IVFFlat) | you already run Postgres: joins with real data, transactions, filters in SQL; the default for most teams | tune `ef_search` / `lists`; very large scale needs care |
| **Qdrant** | purpose-built store (Rust), HNSW with filter-aware traversal | strong filtered search, payload indexes, self-host or cloud | another system to operate |
| **Weaviate** | open-source store with modules, hybrid (BM25 + vector) built in | hybrid search and schema-rich objects out of the box | operational complexity |
| **Milvus / Zilliz** | distributed store, many index types (IVF, HNSW, DiskANN, GPU) | hundreds of millions to billions of vectors | heavier to run (many components) |
| **Pinecone** | fully managed serverless vector DB | no ops, fast time-to-production | cost at scale, vendor lock-in, data residency |
| **Elasticsearch / OpenSearch** | search engine with `dense_vector`/`knn` + BM25 | you need lexical + vector + filters + aggregations in one system | memory-hungry HNSW; tune shards |

**Do you always need one?** No. Under ~100k vectors a NumPy matrix or a FAISS flat index in memory is exact, fast and one less system. Reach for a store when you need persistence, filtering, concurrent updates, multi-tenancy, or more vectors than one machine's RAM.

## Try it yourself

**Exercise 1.** The IVF rule of thumb is `nlist ≈ sqrt(N)` to `4·sqrt(N)` (here ~140-570). For `nlist` in {32, 128, 512}, find the smallest `nprobe` (a power of 2) that reaches recall@10 ≥ 0.95, and report what fraction of the vectors that setting scans. The fraction scanned is roughly your query cost.

In [ ]:
# Solution — try it yourself first, then run this
for nlist in [32, 128, 512]:
    index = faiss.IndexIVFFlat(faiss.IndexFlatIP(DIM), DIM, nlist, faiss.METRIC_INNER_PRODUCT)
    index.cp.min_points_per_centroid = 5                     # allow 512 cells on 20k points without a warning
    index.train(X)
    index.add(X)
    for nprobe in [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]:
        index.nprobe = min(nprobe, nlist)
        r = recall_at_k(index.search(Q, K)[1], truth)
        if r >= 0.95:
            break
    print(f"nlist={nlist:4d}: nprobe={index.nprobe:3d} reaches recall {r:.3f}, scanning ~{index.nprobe / nlist:.1%} of the vectors")
print("few big cells: each probe is expensive; many small cells: you need many probes plus a bigger coarse search")

**Exercise 2.** HNSW memory vs recall: on the first 5,000 vectors (to keep it quick), build `M` in {4, 16, 32} at `efSearch = 32` and report recall@10 and bytes per vector.

In [ ]:
# Solution — try it yourself first, then run this
X5 = X[:5000]
truth5 = numpy_topk(Q, X5, K)
for M in [4, 16, 32]:
    h = faiss.IndexHNSWFlat(DIM, M, faiss.METRIC_INNER_PRODUCT)
    h.hnsw.efConstruction = 40
    h.add(X5)
    h.hnsw.efSearch = 32
    print(f"M={M:2d}: recall@10 = {recall_at_k(h.search(Q, K)[1], truth5):.3f}, {index_bytes(h) / len(X5):.0f} bytes/vector")
print("more links per node: better recall at the same efSearch, more memory (and a slower build)")

**Exercise 3.** Binary quantisation (`ve33`): keep only the sign of each dimension (64 bits = 8 bytes), search by Hamming distance, then re-score a shortlist of 100 with the exact vectors.

In [ ]:
# Solution — try it yourself first, then run this
Xb, Qb = np.packbits(X > 0, axis=1), np.packbits(Q > 0, axis=1)           # 8 bytes per vector
bin_index = faiss.IndexBinaryFlat(DIM)
bin_index.add(Xb)
_, cand = bin_index.search(Qb, 100)
alone = recall_at_k(cand[:, :K], truth)
rescored_b = np.array([c[np.argsort(-(X[c] @ q))[:K]] for c, q in zip(cand, Q)])
print(f"binary codes alone: recall@10 = {alone:.3f}; + exact re-score of 100: {recall_at_k(rescored_b, truth):.3f}")
print("(64-d is low for binary codes; it works far better on 1024-d+ embeddings)")

## Say it in an interview

- **30 seconds:** "Exact search is a matrix multiply - O(N·d) per query, fine to about a hundred thousand vectors. Beyond that I use ANN: HNSW, a layered neighbour graph searched greedily with an efSearch candidate list, or IVF, k-means cells where I scan the nprobe nearest. Both trade recall for speed, so I measure recall@10 against brute force on a sample of real queries and pick the knee - typically 0.95-0.99. When memory binds, I compress with PQ and re-score a shortlist with exact vectors."
- **Numbers:** cosine = dot on unit vectors, and $\|a-b\|^2 = 2 - 2\cos$; HNSW `M` 16-64, `efSearch` the query-time dial; IVF `nlist ≈ sqrt(N)`-`4·sqrt(N)`, `nprobe` the dial; 768-d float32 = 3 KB per vector, PQ ~96 bytes; 10M vectors ≈ 29 GB raw, 1.2B ≈ 3.4 TB.
- **Filters:** post-filtering silently returns fewer than k; pre-filter (or native filtered ANN) for correctness; brute force over tiny subsets; tenant filters are security, enforced server-side.
- **Traps:** forgetting to normalise for inner product; quoting ANN recall as retrieval quality (0.99 recall of bad embeddings is still bad); IVF centroids drifting as the corpus shifts; HNSW deletes are tombstones that need rebuilds.
- **Follow-ups:** "how do you shard?" (by tenant if you can; otherwise scatter-gather, where p95 becomes the max over shards); "do you need a vector DB?" (not below ~100k vectors; pgvector if you already run Postgres).

## Next

- [Hybrid search and RRF](04_hybrid_search_and_rrf.ipynb) - combine this dense lane with BM25.
- [Reranking](05_reranking.ipynb) - precision on top of the recall you just measured.
- [Semantic search service (coding round)](../17_interview_coding_rounds/04_semantic_search_service.ipynb) · [BM25 from scratch](02_bm25_from_scratch.ipynb)
- Theory: [AI system design concepts](../../ai_system_design_concepts/index.html) (product quantisation drawn) · [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)